[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/coopster-seclusion/sar-optical-pipeline/blob/main/notebooks/01_s2_retrieval.ipynb)

# Notebook 01: Sentinel-2 Retrieval

Retrieves S2 SR Harmonized median composites for all epochs defined in `config.yaml` via Google Earth Engine.  
Applies SCL cloud/shadow/snow masking, computes NDVI/NDWI/NDBI/BSI, and exports one multi-band GeoTIFF per epoch to Google Drive.

**Output:** `data/s2/s2_{year}.tif` — bands: B2 B3 B4 B8 B11 NDVI NDWI NDBI BSI (Float32, 0–1 scaled reflectance)

**Before running:**
1. Commit your AOI GeoJSON files to the repo (`aoi/change_area.geojson`, `aoi/stable_reference.geojson`) and push.
2. Edit `config.yaml` — set `site_name`, `aoi` paths, `epochs`, and `crs`.
3. In Colab → Secrets panel, set `GEE_PROJECT` to your Google Earth Engine cloud project ID.

In [ ]:
# ── Cell 1: run this first, every session ─────────────────────────────────────
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, subprocess, sys

REPO_URL = 'https://github.com/coopster-seclusion/sar-optical-pipeline.git'
REPO_DIR = '/content/sar-optical-pipeline'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

sys.path.insert(0, REPO_DIR)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'geemap', 'rioxarray', 'asf_search', 'hyp3_sdk',
                'rasterstats', 'pyogrio'], check=True)

import warnings
import numpy as np
import rioxarray as rxr
import geopandas as gpd
import matplotlib.pyplot as plt
import ee, geemap

from pipeline.env import setup, repo_root
from pipeline.validate import validate_config
from pipeline.utils import resolve_crs
import pipeline.s2 as s2_lib

cfg, DATA, OUT = setup(colab=True)
validate_config(cfg, repo_root())

SITE    = cfg['site_name']
DATA_S2 = DATA / 's2'
OUT_FIGS = OUT / 'figures'

print(f'Site        : {SITE}')
print(f'Data → S2   : {DATA_S2}')
print(f'Epochs      : {[e["year"] for e in cfg["epochs"]]}')

## Authenticate GEE and load AOI

In [ ]:
# ── Cell 2: authenticate GEE and load AOI ─────────────────────────────────────
# GEE_PROJECT must be set in Colab Secrets (key icon in the left sidebar).
ee.Authenticate()
ee.Initialize(project=userdata.get('GEE_PROJECT'))

ROOT     = repo_root()
aoi_path = ROOT / cfg['aoi']['change_area']
aoi_gdf  = gpd.read_file(aoi_path).to_crs('EPSG:4326')
aoi_ee   = geemap.geopandas_to_ee(aoi_gdf)
crs      = resolve_crs(cfg, aoi_path)

print(f'AOI file    : {aoi_path}')
print(f'AOI bounds  : {aoi_gdf.total_bounds.round(4)}  (W S E N)')
print(f'Output CRS  : {crs}')

## Export composites to Google Drive

Files land in `My Drive/{site_name}_s2/`. Cell 5 moves them to `data/s2/` after tasks complete.

Monitor progress at: https://code.earthengine.google.com/tasks

In [ ]:
# ── Cell 3: submit GEE export tasks ───────────────────────────────────────────
DATA_S2.mkdir(parents=True, exist_ok=True)
drive_folder = f'{SITE}_s2'
export_tasks = []

for epoch in cfg['epochs']:
    year     = epoch['year']
    out_path = DATA_S2 / f's2_{year}.tif'

    if out_path.exists():
        print(f'  {year}: already in data dir — skipping')
        continue

    col_size = (
        ee.ImageCollection(cfg['sentinel2']['collection'])
        .filterBounds(aoi_ee.geometry())
        .filterDate(epoch['date_start'], epoch['date_end'])
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cfg['sentinel2']['cloud_pct_max']))
        .size()
        .getInfo()
    )
    print(f'  {year}: {col_size} images pass cloud filter', end='  ', flush=True)

    if col_size == 0:
        print(f'\n  WARNING {year}: no images — raise sentinel2.cloud_pct_max or widen date window')
        continue

    composite = s2_lib.build_s2_composite(
        aoi_ee.geometry(), epoch['date_start'], epoch['date_end'], cfg
    )
    task = s2_lib.export_epoch(composite, year, aoi_ee.geometry(), crs, drive_folder)
    export_tasks.append((year, task))
    print('→ task started')

print(f'\n{len(export_tasks)} task(s) submitted')
print(f'Export folder in Drive: {drive_folder}')

In [ ]:
# ── Cell 4: check task status (re-run to refresh) ─────────────────────────────
for year, task in export_tasks:
    print(f'  {year}: {task.status().get("state", "UNKNOWN")}')

In [ ]:
# ── Cell 5: move exported files to data/s2/ ───────────────────────────────────
# Run after all tasks show COMPLETED in Cell 4.
import shutil, glob as _glob

source_dir = f'/content/drive/MyDrive/{drive_folder}'
DATA_S2.mkdir(parents=True, exist_ok=True)

tif_files = _glob.glob(f'{source_dir}/s2_*.tif')
if not tif_files:
    print(f'No TIF files found in {source_dir}')
    print('Wait for GEE tasks to complete (check Cell 4) before running this cell.')
else:
    for src in tif_files:
        dst = DATA_S2 / os.path.basename(src)
        shutil.move(src, str(dst))
        print(f'  Moved: {os.path.basename(src)}')
    print(f'\nAll files moved to {DATA_S2}')

## Verification

Run after exports complete and files are in `data/s2/`.

In [ ]:
# ── Cell 6: verify first epoch ────────────────────────────────────────────────
BAND_NAMES = ['B2', 'B3', 'B4', 'B8', 'B11', 'NDVI', 'NDWI', 'NDBI', 'BSI']
first_year  = cfg['epochs'][0]['year']
sample_path = DATA_S2 / f's2_{first_year}.tif'

if not sample_path.exists():
    print(f'FILE NOT FOUND: {sample_path}')
    print('GEE tasks may still be running. Check Cell 4, then run Cell 5 to move files.')
else:
    ds = rxr.open_rasterio(sample_path, masked=True)
    ds = ds.assign_coords(band=BAND_NAMES)

    def pct_stretch(arr, low=2, high=98):
        lo, hi = np.nanpercentile(arr, [low, high])
        return np.clip((arr - lo) / (hi - lo), 0, 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    rgb = np.stack([
        pct_stretch(ds.sel(band='B4').values),
        pct_stretch(ds.sel(band='B3').values),
        pct_stretch(ds.sel(band='B2').values),
    ], axis=-1)
    axes[0].imshow(rgb)
    axes[0].set_title(f'True colour {first_year} — {SITE}')
    axes[0].axis('off')

    ndvi = ds.sel(band='NDVI').values
    im   = axes[1].imshow(ndvi, cmap='RdYlGn', vmin=-1, vmax=1)
    axes[1].set_title(f'NDVI {first_year} — {SITE}')
    axes[1].axis('off')
    plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

    plt.tight_layout()
    OUT_FIGS.mkdir(parents=True, exist_ok=True)
    save_path = OUT_FIGS / f's2_{first_year}_preview.png'
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

    print(f'Shape  : {ds.shape}')
    print(f'CRS    : {ds.rio.crs}')
    print(f'Bounds : {[round(x, 4) for x in ds.rio.bounds()]}')
    print(f'Saved  : {save_path}')